# Lab 07: Time Series Decomposition

รายวิชา `229351: Statistical Learning for Data Science 1`

Lab นี้ใช้อนุกรม `elecequip` ชุดเดียวกับที่ใช้ในบทเรียน Lec 07 ทุกกราฟ ตัวเลขทุกตัวที่ได้จึงตรงกับตัวเลขบนสไลด์ ถ้าค่าที่ได้ไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด ไม่ใช่เรื่องของการปัดเศษ

## เป้าหมาย

1. อ่านอนุกรมเวลาเข้ามาเป็น `pandas.Series` ที่มี `DatetimeIndex` รายเดือน และตรวจความสม่ำเสมอของดัชนีเวลา
2. แยก trend, seasonality และ cycle ออกจากกันได้
3. คำนวณ `m`-MA และ `2 x m`-MA ด้วยมือ และอธิบายค่าที่หายไปที่ปลายทั้งสองข้าง
4. คำนวณ classical additive decomposition ครบสี่ขั้นตอน
5. เลือกระหว่าง additive และ multiplicative จากหลักฐานเชิงตัวเลข
6. วัด `F_T` และ `F_S` จาก STL
7. สร้าง decomposition forecast และเทียบกับ benchmark methods บน train/test split

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose, STL

pd.set_option("display.float_format", lambda value: f"{value:.6g}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

# ข้อมูลถูก pin ไว้ที่ commit หนึ่งโดยเฉพาะ เพื่อให้คำตอบเชิงตัวเลขไม่เปลี่ยนตามเวลา
PIN = "f79a6b0ea4203ea83a8c3f39ea046af81066d5d9"
BASE = f"https://raw.githubusercontent.com/skitimo/ds351-data/{PIN}"
print(BASE)

https://raw.githubusercontent.com/skitimo/ds351-data/f79a6b0ea4203ea83a8c3f39ea046af81066d5d9


## 1. ข้อมูลและดัชนีเวลา

`elecequip` คือดัชนีคำสั่งซื้อใหม่ในอุตสาหกรรมผลิตอุปกรณ์ไฟฟ้าของกลุ่มประเทศยูโร 17 ประเทศ รายเดือน ฐาน 2005 = 100

ไฟล์มีสาม column คือเลขลำดับที่ไม่มีชื่อ (`Unnamed: 0`), `time` ในรูปแบบ `YYYY-MM` และ `value`

ขั้นตอนแรกของงานอนุกรมเวลาทุกงานคือทำให้ **ดัชนีเวลาถูกต้อง** ก่อน เพราะ `rolling`, `resample` และ `seasonal_decompose` ล้วนถือว่าแถวที่อยู่ติดกันห่างกันหนึ่งคาบเสมอ ถ้าดัชนีผิดหรือมีเดือนขาดหาย ฟังก์ชันเหล่านี้จะ **ไม่ error** แต่จะให้คำตอบที่ผิดเงียบ ๆ

In [2]:
raw = pd.read_csv(f"{BASE}/elecequip.csv")
display(raw.head())
print(raw.dtypes)

,Unnamed: 0,time,value
0,1,1996-01,79.35
1,2,1996-02,75.78
2,3,1996-03,86.32
3,4,1996-04,72.6
4,5,1996-05,74.86


Unnamed: 0      int64
time           object
value         float64
dtype: object


In [3]:
# TODO: สร้าง Series ชื่อ elec ที่มี DatetimeIndex รายเดือน (ต้นเดือน) จาก column `time`
#       คำแนะนำ: pd.PeriodIndex(..., freq="M").to_timestamp() แปลง "1996-01" เป็น 1996-01-01
elec_index = ...
elec = ...

print(elec.head())
print("n =", len(elec))

AttributeError: 'ellipsis' object has no attribute 'head'

In [ ]:
# ตรวจว่าดัชนีเวลาครบและสม่ำเสมอ
expected = pd.date_range(elec.index[0], elec.index[-1], freq="MS")
print("span:", elec.index[0].strftime("%Y-%m"), "->", elec.index[-1].strftime("%Y-%m"))
print("mean:", round(float(elec.mean()), 2))
print("ดัชนีเวลาครบทุกเดือน:", elec.index.equals(expected))

assert len(elec) == 195
assert elec.index.equals(expected), "มีเดือนขาดหาย หรือดัชนีเวลาไม่ได้เรียงเป็นรายเดือน"
assert abs(float(elec.mean()) - 95.69) < 0.01

### เขียนคำตอบ ส่วนที่ 1

1. เหตุใดการตรวจว่าดัชนีเวลาครบทุกเดือนจึงต้องทำ **ก่อน** เรียก `rolling` หรือ `seasonal_decompose`
2. ถ้าข้อมูลขาดไป 3 เดือนกลางอนุกรม แล้วเราเรียก `rolling(12).mean()` โดยไม่ตรวจ ผลลัพธ์จะผิดอย่างไร

_เขียนคำตอบที่นี่_

## 2. รูปแบบที่ต้องมองหาในกราฟ

**trend** คือการเคลื่อนที่ระยะยาว ขึ้นหรือลง ไม่จำเป็นต้องเป็นเส้นตรง

**seasonality** คือรูปแบบที่ซ้ำด้วย **คาบที่คงที่และทราบล่วงหน้า** เช่น 12 เดือน

**cycle** คือการขึ้นลงที่ไม่มีคาบคงที่ ความยาวไม่แน่นอน มักสัมพันธ์กับภาวะเศรษฐกิจ ใน `elecequip` การตกอย่างรุนแรงช่วงปี 2008-2009 คือ cycle ไม่ใช่ seasonality

### Exercise 1: อ่านรูปแบบจากกราฟสี่ชนิด (12 คะแนน)

In [ ]:
fig, ax = plt.subplots()
ax.plot(elec.index, elec.to_numpy(), linewidth=1)
ax.set_title("elecequip: new orders index, 1996-01 to 2012-03")
ax.set_xlabel("time"); ax.set_ylabel("index (2005 = 100)")
plt.show()

In [ ]:
# TODO: seasonal plot -- หนึ่งเส้นต่อหนึ่งปี แกน x คือเดือน 1-12
#       คำแนะนำ: elec.groupby(elec.index.year) แล้ววนพล็อตทีละปี
fig, ax = plt.subplots()
...
ax.set_xticks(range(1, 13))
ax.set_title("Seasonal plot: one line per year")
ax.set_xlabel("month"); ax.set_ylabel("index")
plt.show()

In [ ]:
# TODO: subseries plot -- แยกเป็น 12 แผง แผงละหนึ่งเดือน พร้อมเส้นค่าเฉลี่ยของเดือนนั้น
#       คำแนะนำ: elec[elec.index.month == month] เลือกข้อมูลของเดือนหนึ่ง, ใช้ ax.axhline สำหรับค่าเฉลี่ย
fig, axes = plt.subplots(1, 12, figsize=(13, 3), sharey=True)
for month, ax in zip(range(1, 13), axes):
    ...
    ax.set_title(month, fontsize=8); ax.set_xticks([])
axes[0].set_ylabel("index")
fig.suptitle("Subseries plot: each panel is one month, red line is that month's mean")
plt.show()

In [ ]:
# TODO: lag plot -- วาด y_t เทียบกับ y_{t-k} สำหรับ k = 1, 3, 6, 12 และคำนวณ correlation
#       คำแนะนำ: elec.shift(lag) เลื่อนอนุกรม, elec.corr(...) คืนค่า correlation
lags = [1, 3, 6, 12]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
lag_corr = {}
for lag, ax in zip(lags, axes):
    lag_corr[lag] = ...
    ...
    ax.set_title(f"lag {lag}: r = {lag_corr[lag]:+.3f}", fontsize=9)
    ax.set_xlabel(f"y(t-{lag})")
axes[0].set_ylabel("y(t)")
plt.show()
print(lag_corr)

### เขียนคำตอบ Exercise 1

1. จาก seasonal plot หรือ subseries plot อนุกรมนี้มีคาบ `m` เท่ากับเท่าใด และดูจากอะไร
2. เดือนใดต่ำผิดปกติอย่างเห็นได้ชัดทุกปี และน่าจะเป็นเพราะอะไร
3. การตกช่วงปี 2008-2009 เป็น seasonality หรือ cycle เพราะอะไร
4. ถ้าอนุกรมมี seasonality คาบ 12 เดือน เรามักคาดว่า `lag_corr[12]` จะสูงกว่า `lag_corr[6]` ตรวจค่าที่คำนวณได้จริง แล้วตอบว่าตรงกับที่คาดหรือไม่ ถ้าไม่ตรง อะไรคือสาเหตุ และต้องทำอะไรกับอนุกรมก่อน lag correlation จึงจะสะท้อน seasonality ได้ชัด

_เขียนคำตอบที่นี่_

## 3. Moving average: เครื่องมือประมาณ trend

`m`-MA ที่ตำแหน่ง `t` คือค่าเฉลี่ยของ `m` ค่าที่อยู่กึ่งกลางที่ `t`

$$\hat{T}_t = \frac{1}{m}\sum_{j=-k}^{k} y_{t+j}, \qquad m = 2k+1$$

เมื่อ `m` เป็นเลขคี่ หน้าต่างจะสมมาตรรอบ `t` พอดี แต่ seasonality รายเดือนมีคาบ `m = 12` ซึ่งเป็น **เลขคู่** หน้าต่างจึงไม่สมมาตร วิธีแก้คือทำ MA สองชั้น เรียกว่า `2 x m`-MA

### Exercise 2: `m`-MA และ `2 x m`-MA (18 คะแนน)

In [ ]:
# TODO: เขียนฟังก์ชัน moving_average(series, m) ด้วย numpy โดยไม่ใช้ .rolling
#       คืน Series ความยาวเท่าเดิม โดยตำแหน่งที่คำนวณไม่ได้ให้เป็น np.nan
#       สำหรับ m เป็นเลขคี่ ให้ k = m // 2 แล้วเฉลี่ย values[t-k : t+k+1]
def moving_average(series: pd.Series, m: int) -> pd.Series:
    values = series.to_numpy(dtype=float)
    n = len(values)
    out = np.full(n, np.nan)
    k = m // 2
    for t in range(k, n - k):
        out[t] = ...
    return pd.Series(out, index=series.index)

ma5_manual = moving_average(elec, 5)
ma5_pandas = elec.rolling(5, center=True).mean()

comparison = pd.DataFrame({"value": elec, "5-MA manual": ma5_manual, "5-MA pandas": ma5_pandas})
display(comparison.head(10))

assert np.allclose(ma5_manual.to_numpy(), ma5_pandas.to_numpy(), equal_nan=True)
assert abs(ma5_manual.iloc[2] - 77.78) < 0.01, "1996-03 ควรได้ 77.78"

In [ ]:
# TODO: สร้าง 2x12-MA โดยทำ 12-MA ก่อน แล้วทำ 2-MA ทับอีกชั้นหนึ่ง
#       คำแนะนำ: .rolling(12, center=True).mean() แล้ว .rolling(2).mean().shift(-1)
ma12 = ...
ma2x12 = ...

lost_head = int(ma2x12.iloc[:12].isna().sum())
lost_tail = int(ma2x12.iloc[-12:].isna().sum())
print("ค่าที่หายไปด้านหน้า:", lost_head)
print("ค่าที่หายไปด้านท้าย:", lost_tail)

assert lost_head == 6 and lost_tail == 6

In [ ]:
fig, ax = plt.subplots()
ax.plot(elec.index, elec.to_numpy(), linewidth=0.8, alpha=0.55, label="elecequip")
ax.plot(ma2x12.index, ma2x12.to_numpy(), linewidth=1.8, color="crimson", label="2x12-MA")
ax.legend(); ax.set_title("2x12-MA removes the seasonal pattern and leaves the trend-cycle")
plt.show()

### เขียนคำตอบ Exercise 2

1. เหตุใด `2 x 12`-MA จึงเสียค่าไป **6** ตัวที่ปลายแต่ละข้าง ไม่ใช่ 12 ตัว
2. ถ้าใช้ `12`-MA เฉย ๆ กับข้อมูลรายเดือน หน้าต่างจะไม่สมมาตรอย่างไร และ `2 x 12`-MA แก้ปัญหานั้นอย่างไร
3. การที่ trend ที่ประมาณได้ขาดหายไป 6 เดือนสุดท้าย จะเป็นปัญหาอย่างไรเมื่อเราต้องการพยากรณ์อนาคต

_เขียนคำตอบที่นี่_

## 4. Classical decomposition

แบบ additive สมมติว่า $y_t = S_t + T_t + R_t$ และคำนวณได้ด้วยมือครบทั้งสี่ขั้นตอน

1. ประมาณ trend-cycle $\hat{T}_t$ ด้วย `2 x m`-MA
2. หา detrended series $y_t - \hat{T}_t$
3. เฉลี่ย detrended series แยกตามเดือน จะได้ seasonal index ดิบของแต่ละเดือน
4. ปรับ seasonal index ให้ผลรวมเป็นศูนย์ แล้ว $\hat{R}_t = y_t - \hat{T}_t - \hat{S}_t$

### Exercise 3: คำนวณ classical additive decomposition ด้วยมือ (18 คะแนน)

In [ ]:
# TODO: ทำสี่ขั้นตอนข้างต้น ให้ได้ seasonal_index เป็น Series ที่ index คือเดือน 1-12
#       ขั้นที่ 2: detrended = elec - ma2x12
#       ขั้นที่ 3: เฉลี่ยแยกตามเดือนด้วย .groupby(detrended.index.month).mean()
#       ขั้นที่ 4: ลบค่าเฉลี่ยของ seasonal index ออก เพื่อให้ผลรวมเป็นศูนย์
detrended = ...
seasonal_raw = ...
seasonal_index = ...

display(seasonal_index.round(2).to_frame("S_hat"))
print("ผลรวม:", round(float(seasonal_index.sum()), 10))

assert abs(float(seasonal_index.sum())) < 1e-9, "seasonal index ต้องรวมกันได้ศูนย์"
assert abs(seasonal_index[8] - (-16.87)) < 0.01, "เดือน 8 ควรได้ -16.87"

In [ ]:
# ตรวจกับ statsmodels
decomposition = seasonal_decompose(elec, model="additive", period=12)
sm_index = decomposition.seasonal.groupby(decomposition.seasonal.index.month).first()

check = pd.DataFrame({"manual": seasonal_index, "statsmodels": sm_index})
check["diff"] = check["manual"] - check["statsmodels"]
display(check.round(6))

assert np.allclose(seasonal_index.to_numpy(), sm_index.to_numpy()), "ต้องตรงกับ statsmodels"

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, series, label in zip(
    axes,
    [elec, decomposition.trend, decomposition.seasonal, decomposition.resid],
    ["observed", "trend", "seasonal", "remainder"],
):
    ax.plot(series.index, series.to_numpy(), linewidth=1)
    ax.set_ylabel(label)
axes[0].set_title("Classical additive decomposition of elecequip")
plt.show()

### เขียนคำตอบ Exercise 3

1. เหตุใดขั้นตอนที่ 4 จึงต้องปรับ seasonal index ให้ผลรวมเป็นศูนย์
2. `seasonal_index` ของเดือน 8 ต่ำกว่าเดือนอื่นมาก ให้ตีความในเชิงข้อมูลจริงว่าหมายถึงอะไร
3. classical decomposition สมมติว่า seasonal component **คงที่ตลอดทั้งอนุกรม** ข้อสมมตินี้สมเหตุสมผลกับ `elecequip` หรือไม่ และดูจากอะไร

_เขียนคำตอบที่นี่_

## 5. Additive หรือ multiplicative

ถ้าขนาดของการแกว่งตามฤดูกาล **โตขึ้นตามระดับของอนุกรม** แบบจำลองที่เหมาะคือ multiplicative คือ $y_t = S_t \times T_t \times R_t$

ข้อนี้ตัดสินด้วยหลักฐาน ไม่ใช่ด้วยความเคยชิน วิธีวัดที่ตรงไปตรงมาคือดู correlation ระหว่าง **ระดับเฉลี่ยรายปี** กับ **ช่วงกว้างสูงสุด-ต่ำสุดรายปี** ถ้าเป็นบวกสูง แปลว่าแกว่งกว้างขึ้นเมื่อระดับสูงขึ้น

### Exercise 4: ตัดสิน additive เทียบ multiplicative จากตัวเลข (12 คะแนน)

In [ ]:
air = pd.Series(
    (a := pd.read_csv(f"{BASE}/AirPassengers.csv"))["Passengers"].to_numpy(dtype=float),
    index=pd.PeriodIndex(a["Month"], freq="M").to_timestamp(), name="AirPassengers")
cars = pd.Series(
    (c := pd.read_csv(f"{BASE}/CarSales.csv"))["Sales"].to_numpy(dtype=float),
    index=pd.PeriodIndex(c["Month"], freq="M").to_timestamp(), name="CarSales")
print(len(air), len(cars))

In [ ]:
# TODO: เขียน swing_corr(series) คืน correlation ระหว่างค่าเฉลี่ยรายปี กับ (max - min) รายปี
#       นับเฉพาะปีที่มีข้อมูลครบ 12 เดือน
#       คำแนะนำ: grouped = series.groupby(series.index.year) แล้วใช้ .mean(), .max(), .min(), .size()
#                ใช้ np.corrcoef(a, b)[0, 1] คำนวณ correlation
def swing_corr(series: pd.Series) -> float:
    grouped = series.groupby(series.index.year)
    level = ...
    amplitude = ...
    complete = grouped.size() == 12
    return ...

evidence = pd.Series({s.name: swing_corr(s) for s in (air, cars, elec)}, name="corr(level, swing)")
display(evidence.round(3).to_frame())

assert abs(evidence["AirPassengers"] - 0.991) < 0.01
assert abs(evidence["elecequip"] - 0.409) < 0.01

In [ ]:
# multiplicative decomposition เหมาะกับ AirPassengers
air_mul = seasonal_decompose(air, model="multiplicative", period=12)
air_index = air_mul.seasonal.groupby(air_mul.seasonal.index.month).first()
display(air_index.round(3).to_frame("S_hat (multiplicative)"))
print("ผลรวม:", round(float(air_index.sum()), 6), " (ควรเท่ากับ m = 12)")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, series, title in [(axes[0], air, "AirPassengers: swing grows with level"),
                          (axes[1], elec, "elecequip: swing roughly constant")]:
    ax.plot(series.index, series.to_numpy(), linewidth=1); ax.set_title(title, fontsize=10)
plt.show()

assert abs(float(air_index.sum()) - 12.0) < 1e-6

### เขียนคำตอบ Exercise 4

1. จากตาราง `evidence` อนุกรมใดควรใช้ multiplicative decomposition และอนุกรมใดควรใช้ additive
2. `CarSales` ได้ค่า `+0.433` ซึ่งอยู่กลาง ๆ ถ้าดูแต่กราฟอาจเดาว่าเป็น multiplicative ตัวเลขนี้บอกอะไรที่กราฟไม่ได้บอก
3. เหตุใด seasonal index ของ multiplicative model จึงรวมกันได้ `m = 12` แทนที่จะเป็นศูนย์
4. ถ้าใช้ multiplicative model กับอนุกรมที่มีค่าเป็นศูนย์หรือติดลบ จะเกิดปัญหาอะไร

_เขียนคำตอบที่นี่_

## 6. STL และการวัดความแรงของ trend กับ seasonality

classical decomposition บังคับให้ seasonal component คงที่ตลอดอนุกรม **STL** (Seasonal-Trend decomposition using Loess) ยอมให้ seasonal component ค่อย ๆ เปลี่ยนรูปได้

จาก decomposition ใด ๆ วัดความแรงได้ด้วย

$$F_T = \max\left(0,\ 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(T_t + R_t)}\right), \qquad F_S = \max\left(0,\ 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(S_t + R_t)}\right)$$

ทั้งสองค่าอยู่ระหว่าง 0 ถึง 1 ยิ่งเข้าใกล้ 1 ยิ่งแรง

### Exercise 5: `F_T` และ `F_S` จาก STL (14 คะแนน)

**สำคัญ:** ต้องเรียก `STL(..., robust=True)` ค่า default คือ `robust=False` ซึ่งจะให้คำตอบที่ไม่ตรงกับสไลด์

In [ ]:
# TODO: เขียน strengths(series, period=12) คืน (F_T, F_S) โดยใช้ STL(..., robust=True)
#       คำแนะนำ: result = STL(series, period=period, robust=True).fit()
#                ใช้ result.trend, result.seasonal, result.resid และ np.var
def strengths(series: pd.Series, period: int = 12) -> tuple[float, float]:
    result = STL(series, period=period, robust=True).fit()
    remainder = result.resid
    f_t = ...
    f_s = ...
    return float(f_t), float(f_s)

table = pd.DataFrame(
    [dict(zip(("F_T", "F_S"), strengths(s))) for s in (elec, air, cars)],
    index=[s.name for s in (elec, air, cars)])
display(table.round(3))

assert abs(table.loc["elecequip", "F_T"] - 0.890) < 0.005
assert abs(table.loc["elecequip", "F_S"] - 0.837) < 0.005

In [ ]:
# STL ยอมให้ seasonal เปลี่ยนรูปได้ classical ไม่ยอม -- เห็นชัดที่สุดบน AirPassengers
air_stl = STL(air, period=12, robust=True).fit()
air_classical = seasonal_decompose(air, model="additive", period=12)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for year in (1950, 1954, 1958, 1960):
    for ax, comp in zip(axes, (air_classical.seasonal, air_stl.seasonal)):
        one = comp[comp.index.year == year]
        ax.plot(one.index.month, one.to_numpy(), marker="o", markersize=3, label=str(year))
axes[0].set_title("classical: identical every year", fontsize=10)
axes[1].set_title("STL: the shape grows", fontsize=10)
axes[0].set_xticks(range(1, 13)); axes[1].set_xticks(range(1, 13))
axes[1].legend(fontsize=8)
plt.show()

### เขียนคำตอบ Exercise 5

1. `elecequip` ได้ `F_T` และ `F_S` เท่าใด และตีความว่าอนุกรมนี้ trend แรงหรือ seasonality แรงกว่ากัน
2. เปรียบเทียบ `F_T` ของทั้งสามอนุกรม อนุกรมใด trend แรงที่สุด และสอดคล้องกับกราฟหรือไม่
3. จากกราฟสุดท้าย STL ต่างจาก classical decomposition อย่างไร และความต่างนี้สำคัญกับ `AirPassengers` เพราะอะไร
4. ถ้าอนุกรมหนึ่งมี trend เกือบแบนแต่ seasonality ชัดมาก `F_T` และ `F_S` ควรออกมาเป็นอย่างไร

_เขียนคำตอบที่นี่_

## 7. การพยากรณ์ด้วย decomposition และการวัดความแม่นยำ

decomposition ไม่ได้ใช้เพื่อดูอย่างเดียว แต่ใช้พยากรณ์ได้ วิธีที่ตรงที่สุดคือ

1. แยก train ออกเป็น $\hat{S}_t$ และ seasonally adjusted series $A_t = y_t - \hat{S}_t$
2. พยากรณ์ $A_t$ แบบ naive คือใช้ค่าล่าสุดค่าเดียวตลอดช่วงที่พยากรณ์
3. บวก seasonal index ของเดือนนั้นกลับเข้าไป

เราจะเทียบกับ **benchmark methods** สี่วิธี ได้แก่ Mean, Naive, Seasonal naive และ Drift ถ้าวิธีที่ซับซ้อนกว่าชนะ benchmark ไม่ได้ ก็ไม่มีเหตุผลที่จะใช้มัน

### Exercise 6: train/test split และการเทียบ 5 วิธี (18 คะแนน)

In [ ]:
# แบ่ง train/test แบบเดียวกับที่ใช้ในสไลด์ -- test คือ 12 เดือนสุดท้าย
train = elec.loc[:"2011-03"]
test = elec.loc["2011-04":]
print(f"train: {len(train)} obs, {train.index[0]:%Y-%m} -> {train.index[-1]:%Y-%m}")
print(f"test : {len(test)} obs, {test.index[0]:%Y-%m} -> {test.index[-1]:%Y-%m}")

assert len(train) == 183 and len(test) == 12

In [ ]:
# TODO: สร้าง benchmark forecasts ทั้งสี่วิธี ความยาว 12 ค่า สำหรับช่วง test
#       Mean   : ค่าเฉลี่ยของ train ทุกค่า
#       Naive  : ค่าสุดท้ายของ train ทุกค่า
#       SNaive : 12 ค่าสุดท้ายของ train ตามลำดับเดิม
#       Drift  : y_T + h * (y_T - y_1) / (T - 1)
h = np.arange(1, 13)
f_mean = ...
f_naive = ...
f_snaive = ...
f_drift = ...

print("Mean  :", np.round(f_mean[:3], 2))
print("Naive :", np.round(f_naive[:3], 2))
print("SNaive:", np.round(f_snaive[:3], 2))
print("Drift :", np.round(f_drift[:3], 2))

In [ ]:
# decomposition forecast
#
# อ่านให้ดี: anchor ที่ใช้ ไม่ใช่ ค่าสุดท้ายของ seasonally adjusted series
# 2x12-MA เสีย trend ไป 6 เดือนท้าย ดังนั้นค่า A_t ที่ยัง "มี trend รองรับ" ตัวสุดท้าย
# อยู่ที่ 2010-09 ไม่ใช่ 2011-03 การใช้ .iloc[-1] จะได้ RMSE 5.66 ซึ่งไม่ตรงกับสไลด์
train_dec = seasonal_decompose(train, model="additive", period=12)
adjusted = train - train_dec.seasonal
anchor = float(adjusted[train_dec.trend.notna()].iloc[-1])
anchor_date = adjusted[train_dec.trend.notna()].index[-1]
print(f"anchor = {anchor:.3f} ที่ {anchor_date:%Y-%m}")
print(f"ถ้าใช้ .iloc[-1] จะได้ {float(adjusted.iloc[-1]):.3f} ที่ {adjusted.index[-1]:%Y-%m}  <- ผิด")

# TODO: บวก seasonal index ของเดือนนั้นกลับเข้าไปกับ anchor
#       คำแนะนำ: วนตาม test.index แล้วใช้ seasonal_by_month[stamp.month]
seasonal_by_month = train_dec.seasonal.groupby(train_dec.seasonal.index.month).first()
f_decomp = ...
print(np.round(f_decomp[:3], 2))

assert abs(anchor - 88.832) < 0.01

In [ ]:
# TODO: เขียนฟังก์ชัน rmse, mae, mape แล้วสร้างตารางเปรียบเทียบทั้ง 5 วิธี
#       MAPE รายงานเป็นเปอร์เซ็นต์
def rmse(actual, predicted): return ...
def mae(actual, predicted):  return ...
def mape(actual, predicted): return ...

y_true = test.to_numpy()
methods = {"Mean": f_mean, "Naive": f_naive, "Seasonal naive": f_snaive,
           "Drift": f_drift, "Decomposition": f_decomp}
accuracy = pd.DataFrame(
    [{"RMSE": rmse(y_true, f), "MAE": mae(y_true, f), "MAPE": mape(y_true, f)} for f in methods.values()],
    index=list(methods))
display(accuracy.round(2).sort_values("RMSE"))

assert abs(accuracy.loc["Decomposition", "RMSE"] - 4.85) < 0.01
assert abs(accuracy.loc["Seasonal naive", "RMSE"] - 5.11) < 0.01

In [ ]:
fig, ax = plt.subplots()
recent = elec.loc["2008-01":]
ax.plot(recent.index, recent.to_numpy(), color="black", linewidth=1.2, label="actual")
for label, forecast in methods.items():
    ax.plot(test.index, forecast, linewidth=1.4, marker="o", markersize=3, label=label)
ax.axvline(train.index[-1], color="grey", linestyle="--", linewidth=1)
ax.legend(fontsize=8, ncol=2); ax.set_title("Forecasts on the 12-month test window")
plt.show()

### เขียนคำตอบ Exercise 6

1. วิธีใดได้ RMSE ต่ำที่สุด และชนะอันดับสองอยู่เท่าไร
2. ผลต่างนั้นมากพอที่จะสรุปว่า decomposition "ดีกว่าอย่างชัดเจน" หรือไม่ ให้เหตุผล
3. `Naive` และ `Drift` แย่กว่าวิธีอื่นมาก เพราะเหตุใด ให้อธิบายโดยอ้างถึง seasonality ของอนุกรมนี้
4. บนสไลด์ เมื่อเปลี่ยนช่วงทดสอบเป็น 27 เดือน อันดับกลับด้าน โดย `Mean` ชนะที่ RMSE 9.07 ขณะที่ decomposition ได้ 14.59 อธิบายว่าเหตุใดความยาวของ horizon จึงเปลี่ยนอันดับได้ขนาดนั้น
5. จากข้อ 4 การสรุปว่า "decomposition เป็นวิธีที่ดีที่สุดสำหรับอนุกรมนี้" เป็นข้อสรุปที่ซื่อสัตย์ต่อหลักฐานหรือไม่ ควรพูดให้ถูกต้องอย่างไร

_เขียนคำตอบที่นี่_

## Checklist ก่อนส่ง

- [ ] โหลดข้อมูลจาก URL ที่ pin ไว้ และได้ 195 observations
- [ ] ดัชนีเวลาเป็นรายเดือนครบถ้วน ไม่มีเดือนขาดหาย
- [ ] `moving_average` ที่เขียนเองให้ผลตรงกับ `rolling(5, center=True)`
- [ ] `2 x 12`-MA เสียค่าไป 6 ตัวที่ปลายแต่ละข้าง และอธิบายได้ว่าทำไม
- [ ] seasonal index ที่คำนวณด้วยมือรวมกันได้ศูนย์ และตรงกับ `seasonal_decompose`
- [ ] อ้างตาราง `evidence` เมื่อเลือกระหว่าง additive และ multiplicative
- [ ] เรียก `STL` ด้วย `robust=True` และได้ `F_T = 0.890`, `F_S = 0.837`
- [ ] train มี 183 observations, test มี 12 observations
- [ ] ใช้ anchor ที่ `2010-09` ไม่ใช่ `2011-03` และได้ decomposition RMSE = 4.85
- [ ] ตอบคำถามทุกข้อในทุก markdown cell ที่ขึ้นต้นด้วย "เขียนคำตอบ"
- [ ] restart kernel แล้ว run all ผ่านตลอดโดยไม่มี error
- [ ] เปลี่ยนชื่อไฟล์เป็น `lab07_studentid.ipynb` ก่อนส่ง